# 70 — Integrated Evidence Inventory

Collects environmental, documentary, and current full-run outputs into one inventory.

In [ ]:
import os, io, re, json, hashlib, platform, sys, zipfile
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def safe_read_parquet(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        import pyarrow.parquet as pq
        df = pq.read_table(p).to_pandas()
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def clean_facility(fn: str) -> str:
    s = re.sub(r"\.xlsm$|\.xlsx$|\.pdf$", "", str(fn), flags=re.I)
    s = re.sub(r"^[A-Z0-9]+\s+", "", s)
    s = re.sub(r"\s+Annual Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Annual Performance Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Report 2024$", "", s, flags=re.I)
    s = re.sub(r"\s+2024$", "", s)
    return s.strip()

run_cfg = load_yaml(CONFIGS / "run.yml")
phase70_cfg = load_yaml(CONFIGS / "phase70.yml")
doc_cfg = load_yaml(CONFIGS / "documentary_sources.yml")

In [ ]:
step = "70_evidence_inventory_plus"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

expected = [
    "outputs/42_multi_site_comparison/site_summary.csv",
    "outputs/43_window_ranking/window_top_candidates.csv",
    "outputs/52_weather_validation_and_fallback_audit/weather_validation_summary.json",
    "outputs/57_contradictions_red_team_audit/contradiction_register.csv",
    "outputs/60_final_integrity_report/final_integrity_report.json",
    "outputs/65_documentary_evidence_fusion/documentary_evidence_site_year.csv",
    "outputs/66_documentary_adjudication_patch/site_adjudication_patched.csv",
    "outputs/67_final_forensic_summary/final_documentary_forensic_summary.json",
    "data_sources/incinerator_report_catalog.csv",
    "data_sources/cems_vocab_hits.csv",
    "data_sources/cems_table_blocks_hits.csv",
    "data_sources/emissions_long.parquet",
]
rows = []
for rel in expected:
    p = PROJECT_ROOT / rel
    rows.append({
        "path": rel,
        "exists": p.exists(),
        "size_bytes": p.stat().st_size if p.exists() else 0,
        "sha256": sha256_file(p) if p.exists() else "",
    })
inv = pd.DataFrame(rows)
inv["status"] = np.where(inv["exists"], "present", "missing")
inv.to_csv(out / "integrated_inventory.csv", index=False)
summary = {
    "present": int(inv["exists"].sum()),
    "missing": int((~inv["exists"]).sum()),
}
write_json(out / "integrated_inventory_summary.json", summary)
manifest = manifest_base(step, [CONFIGS / "phase70.yml", CONFIGS / "documentary_sources.yml"])
add_artifact(manifest, out / "integrated_inventory.csv")
add_artifact(manifest, out / "integrated_inventory_summary.json")
write_json(out / "manifest.json", manifest)
display(inv)